# Day 24：FastAPI Part 2 — SSE 流式输出 + CORS + SQLite 持久化 + Web ChatGPT

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第二阶段收官之战（Day 24）！今天我们将使用 `ChatDatabase`（SQLite 持久化）、FastAPI `StreamingResponse`（SSE 打字机流式接口）、`CORSMiddleware` 与 `frontend/index.html`（Web ChatGPT 克隆版），完成从后端到前端的全栈闭环（`v0.2.0-day24`）。

## 1. 环境初始化与导入 `project_iterations/day_24` 模块
使用 `tempfile.TemporaryDirectory` 创建隔离的临时 SQLite 数据库，并通过 FastAPI `TestClient` 进行无服务器启动的端到端验证。

In [1]:
import json
import sys
import tempfile
from pathlib import Path

from fastapi.testclient import TestClient

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day24_dir = repo_root / "project_iterations" / "day_24"
if str(day24_dir) not in sys.path:
    sys.path.insert(0, str(day24_dir))

from zhishu_agent import ChatDatabase, create_app

tmp_dir = tempfile.TemporaryDirectory()
db_path = Path(tmp_dir.name) / "notebook_demo.db"
app = create_app(db_path=db_path, frontend_dir=day24_dir / "frontend")
client = TestClient(app)

print("健康检查:", client.get("/api/v1/health").json())

健康检查: {'status': 'ok', 'version': 'v0.2.0-day24', 'db_path': '/tmp/tmpck31597m/notebook_demo.db'}


/home/antigravity/ab6539763/-AI-Agent/.venv/lib/python3.11/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 2. 验证 SQLite 持久化仓储（`ChatDatabase`）
演示写入多轮对话、列出历史会话以及恢复指定会话历史。

In [2]:
db = ChatDatabase(db_path)
db.save_message("demo-db", "user", "你好，请问智枢平台支持持久化吗？")
db.save_message("demo-db", "assistant", "支持！已接入 SQLite sessions 与 messages 双表存储。")

print("会话列表:", db.list_sessions())
print("历史消息条数:", len(db.get_history("demo-db")))

会话列表: [{'session_id': 'demo-db', 'title': '你好，请问智枢平台支持持久化吗？', 'created_at': '2026-11-13T05:56:27.434821+00:00', 'updated_at': '2026-11-13T05:56:27.440967+00:00', 'message_count': 2}]
历史消息条数: 2


## 3. 验证 SSE 流式输出（`POST /api/v1/chat/stream`）与 CORS 跨域头
通过 `client.stream(...)` 逐行解析 `data: {...}` SSE 事件，并验证消息自动落库到 SQLite。

In [3]:
# 1) 检查 CORS 预检请求
cors_resp = client.options(
    "/api/v1/chat/stream",
    headers={
        "Origin": "http://localhost:3000",
        "Access-Control-Request-Method": "POST",
    },
)
print("CORS Allow-Origin:", cors_resp.headers.get("access-control-allow-origin"))

# 2) 消费 SSE 流式响应
session_id = "nb-sse-session"
chunks = []
with client.stream(
    "POST",
    "/api/v1/chat/stream",
    json={"session_id": session_id, "message": "请介绍 SSE 流式输出", "delay_ms": 0},
) as resp:
    print("SSE Content-Type:", resp.headers.get("content-type"))
    for line in resp.iter_lines():
        if line.startswith("data: "):
            evt = json.loads(line[6:])
            if evt["type"] == "delta":
                chunks.append(evt["content"])

print(f"共收到 {len(chunks)} 个打字机分块，拼接结果:\n" + "".join(chunks))

# 3) 校验自动持久化结果与前端页面
hist = client.get(f"/api/v1/sessions/{session_id}/history").json()
print("SQLite 已持久化消息数:", len(hist["messages"]))
home_resp = client.get("/")
print("前端页面状态码:", home_resp.status_code, "| 包含 ReadableStream:", "getReader()" in home_resp.text)

tmp_dir.cleanup()

CORS Allow-Origin: *
SSE Content-Type: text/event-stream; charset=utf-8
共收到 47 个打字机分块，拼接结果:
【智枢平台 · 第 1 轮回复】
SSE（Server-Sent Events）基于标准 HTTP 长连接与 `text/event-stream` 协议，由后端通过 `StreamingResponse` 逐块推送 `data: {...}\n\n` 事件，前端再借助 `fetch()` + `response.body.getReader()` 实现逐字打字机渲染。
SQLite 已持久化消息数: 2
前端页面状态码: 200 | 包含 ReadableStream: True
